# 장비 자신의 과거 기준으로 고장 전 분류 (초안)

| 항목 | 내용 |
|---|---|
| 데이터 | `data/processed/step/` (레시피 스텝 구간 하나 = 한 행), 셔터 1만 |
| 동기 | EDA(`eda/eda_step.ipynb`)에서는 각 장비를 **자기 평소 값**과 비교했을 때 냉각 압력 신호가 보였다(같은 장비 안 AUC 0.29~0.37). 모델에서는 **다른 장비들의 평소 값**을 기준으로 써서 신호가 묻혔을 수 있다 |
| 질문 | 장비 자신의 과거 값을 기준으로 하면, 처음 보는 장비에서도 고장 전을 구분할 수 있는가? |
| 상태 | 초안 (튜닝 최소) |

## 1. 방법

**z값 기준 3가지** (모두 레시피·스텝별 평소 구간의 평균·표준편차, 평소 = 고장에서 24시간 넘게 떨어진 구간)

| 기준 | 계산에 쓰는 구간 | 실제 운영에서 |
|---|---|---|
| A. 다른 장비 기준 | 학습 장비들의 전체 평소 구간 | 가능 (지금까지 모델 방식) |
| **B. 장비 자신의 과거** | **그 장비의 처음 60일** 평소 구간 | **가능** (새 장비는 60일 동안 기준을 쌓은 뒤 감시) |
| C. 장비 자신의 전체 기간 | 그 장비의 전체 평소 구간 | 불가능 (미래 정보 포함, 상한선 확인용 = EDA 방식) |

- **평가 구간**: 세 기준 모두 각 장비의 **60일 이후** 구간에서만 학습·평가한다(B의 기준 기간과 겹치지 않게).
- **라벨**: 고장 3종 중 하나라도 TTF ≤ 24시간(또는 48시간)이면 고장 전(1). 기록 끝 24시간 안에서 TTF가 모두 비어 판단할 수 없는 구간은 뺀다.
- **입력**: 누적 카운터를 뺀 센서 13개 × (평균·최소·최대) z값 39개 + 장비별 지난 6시간·24시간 동안의 z 평균 26개 = 65개.
- **모델**: LightGBM. 검증은 지금까지와 같은 장비 단위 4묶음.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next(p for p in (cwd, cwd.parent) if (p / 'data/processed/step').exists())
PROC = ROOT / 'data/processed'

def show(fig, height=480):
    fig.update_layout(template='plotly_white', height=height, font=dict(family='Malgun Gothic, sans-serif'),
                      margin=dict(l=30, r=30, t=75, b=60))
    display({'application/vnd.plotly.v1+json': json.loads(fig.to_json())}, raw=True)

SENSORS = ['IONGAUGEPRESSURE', 'ETCHBEAMVOLTAGE', 'ETCHBEAMCURRENT', 'ETCHSUPPRESSORVOLTAGE', 'ETCHSUPPRESSORCURRENT',
           'FLOWCOOLFLOWRATE', 'FLOWCOOLPRESSURE', 'ETCHGASCHANNEL1READBACK', 'ETCHPBNGASREADBACK',
           'FIXTURETILTANGLE', 'ROTATIONSPEED', 'ACTUALROTATIONANGLE', 'ACTUALSTEPDURATION']
NAMES = {'IONGAUGEPRESSURE': '챔버 압력', 'ETCHBEAMVOLTAGE': '빔 전압', 'ETCHBEAMCURRENT': '빔 전류',
         'ETCHSUPPRESSORVOLTAGE': '억제 전압', 'ETCHSUPPRESSORCURRENT': '억제 전류', 'FLOWCOOLFLOWRATE': '냉각 유량',
         'FLOWCOOLPRESSURE': '냉각 압력', 'ETCHGASCHANNEL1READBACK': '아르곤 유량(소스)', 'ETCHPBNGASREADBACK': '아르곤 유량(PBN)',
         'FIXTURETILTANGLE': '웨이퍼 기울기', 'ROTATIONSPEED': '회전 속도', 'ACTUALROTATIONANGLE': '회전 각도',
         'ACTUALSTEPDURATION': '단계 소요시간'}
STATS = ['mean', 'min', 'max']
FAULTS = ['FlowCool Pressure Dropped Below Limit', 'Flowcool Pressure Too High Check Flowcool Pump', 'Flowcool leak']
TTF = ['TTF_' + f for f in FAULTS]
HOUR, DAY = 3600, 86400
BASE_DAYS = 60
SEED = 0

cols = ['Tool', 'time_start', 'time_end', 'recipe', 'recipe_step', 'shutter', *TTF,
        *[f'{s}_{st}' for s in SENSORS for st in STATS]]
d = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted((PROC / 'step').glob('*.parquet'))], ignore_index=True)
d['Tool'] = d.Tool.astype(str)

# 평소 여부(가장 가까운 고장까지 24시간 초과)와 판단 불가 구간은 셔터와 무관하게 전체에서 계산
faults = pd.read_csv(PROC / 'faults.csv')
faults['Tool'] = faults.Tool.map(lambda t: f'{t[2:]}_{t[:2]}')
fdist = np.full(len(d), np.inf)
for tool, idx in d.groupby('Tool').indices.items():
    ft = np.sort(faults.loc[faults.Tool == tool, 'time'].unique())
    ts, te = d.time_start.to_numpy()[idx], d.time_end.to_numpy()[idx]
    j = np.searchsorted(ft, ts); k = np.searchsorted(ft, te)
    prev = np.where(j > 0, ts - ft[np.maximum(j - 1, 0)], np.inf)
    nxt = np.where(k < len(ft), ft[np.minimum(k, len(ft) - 1)] - te, np.inf)
    fdist[idx] = np.minimum(prev, nxt)
d['normal'] = fdist > DAY
end_t = d.groupby('Tool').time_end.transform('max')
d['censored'] = ((end_t - d.time_end) < DAY) & d[TTF].isna().all(axis=1)
d['start_t'] = d.groupby('Tool').time_start.transform('min')
d['in_base'] = d.time_start < d.start_t + BASE_DAYS * DAY

d = d[d.shutter == 1].copy()
d['_rs'] = d.recipe_step.fillna(-1)
d = d.sort_values(['Tool', 'time_start']).reset_index(drop=True)
d['ttf_min'] = d[TTF].min(axis=1)
print(f'셔터 1 스텝 구간 {len(d):,}행 · 기준 기간(처음 {BASE_DAYS}일) {d.in_base.mean():.1%} · 판단 불가 {d.censored.sum():,}행')

셔터 1 스텝 구간 526,266행 · 기준 기간(처음 60일) 14.3% · 판단 불가 1,212행


## 2. z값 만들기

In [2]:
MEANS = [s + '_mean' for s in SENSORS]

def stats(ref, keys, min_n=10):
    g = ref.groupby(keys)[MEANS]
    n = g.size()
    mu, sd = g.mean(), g.std()
    sd = sd.where((sd > 1e-6) & np.broadcast_to((n >= min_n).to_numpy()[:, None], sd.shape))
    return mu, sd

def to_z(mu, sd, keys):
    key = pd.MultiIndex.from_frame(d[keys])
    m = mu.reindex(key).to_numpy(); s_ = sd.reindex(key).to_numpy()
    out = {}
    for k, s in enumerate(SENSORS):
        for st in STATS:
            out[f'{s}_{st}_z'] = np.clip((d[f'{s}_{st}'].to_numpy() - m[:, k]) / s_[:, k], -10, 10)
    return pd.DataFrame(out, index=d.index).astype('float32')

def add_rolling(z):
    zm = z[[f'{s}_mean_z' for s in SENSORS]].copy()
    zm.index = pd.to_datetime(d.time_start, unit='s')
    parts = []
    for win in ['6h', '24h']:
        r = zm.groupby(d.Tool.to_numpy()).rolling(win, min_periods=3).mean()
        r = r.reset_index(level=0, drop=True)
        r.columns = [c + f'_r{win}' for c in zm.columns]
        parts.append(r.reset_index(drop=True))
    # groupby-rolling은 장비 순서대로 나오며, d가 장비·시각으로 정렬돼 있어 행 순서가 같다
    return pd.concat([z.reset_index(drop=True)] + parts, axis=1).set_index(d.index)

normal_ref = d.normal & ~d.censored
Z_self_past = add_rolling(to_z(*stats(d[normal_ref & d.in_base], ['Tool', 'recipe', '_rs']), ['Tool', 'recipe', '_rs']))
Z_self_full = add_rolling(to_z(*stats(d[normal_ref], ['Tool', 'recipe', '_rs']), ['Tool', 'recipe', '_rs']))
FEATS = list(Z_self_past.columns)

ev_mask = (~d.in_base & ~d.censored).to_numpy()
cov = pd.DataFrame({
    'B. 장비 자신의 과거': [Z_self_past.loc[ev_mask, 'FLOWCOOLPRESSURE_mean_z'].notna().mean()],
    'C. 장비 자신의 전체 기간': [Z_self_full.loc[ev_mask, 'FLOWCOOLPRESSURE_mean_z'].notna().mean()],
}, index=['평가 구간 중 냉각 압력 z 있음']).map(lambda v: f'{v:.1%}')
display(cov)

,B. 장비 자신의 과거,C. 장비 자신의 전체 기간
평가 구간 중 냉각 압력 z 있음,95.6%,98.8%


## 3. 학습과 평가

In [3]:
tools = sorted(d.Tool.unique(), key=lambda t: f'{t[4:]}{t[:3]}')
perm = np.random.default_rng(SEED).permutation(len(tools))
folds = [[tools[i] for i in perm[k::4]] for k in range(4)]

def make_gbm():
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=200,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, is_unbalance=True,
                              random_state=SEED, verbose=-1)

rows, imps, scores = [], [], {}
for W in [24, 48]:
    y_all = (d.ttf_min <= W * HOUR).to_numpy().astype(int)
    for name in ['A. 다른 장비 기준', 'B. 장비 자신의 과거', 'C. 장비 자신의 전체 기간']:
        sc = np.full(len(d), np.nan)
        for k, test_tools in enumerate(folds):
            is_test = d.Tool.isin(test_tools).to_numpy()
            if name.startswith('A'):
                mu, sd = stats(d[normal_ref & ~d.in_base & ~is_test], ['recipe', '_rs'], min_n=30)
                Z = add_rolling(to_z(mu, sd, ['recipe', '_rs']))
            else:
                Z = Z_self_past if name.startswith('B') else Z_self_full
            tr = ev_mask & ~is_test
            te = ev_mask & is_test
            model = make_gbm().fit(Z.loc[tr, FEATS], y_all[tr])
            sc[te] = model.predict_proba(Z.loc[te, FEATS])[:, 1]
            rows.append({'창': f'{W}시간', '기준': name, '묶음': k + 1, 'AUC': roc_auc_score(y_all[te], sc[te]),
                         '평가 구간': int(te.sum()), '고장 전 비율': y_all[te].mean()})
            if W == 24 and name.startswith('B'):
                imps.append(pd.Series(model.booster_.feature_importance('gain'), index=FEATS))
        scores[(W, name)] = sc
        print(f'{W}시간 · {name} 완료', flush=True)
res = pd.DataFrame(rows)

24시간 · A. 다른 장비 기준 완료


24시간 · B. 장비 자신의 과거 완료


24시간 · C. 장비 자신의 전체 기간 완료


48시간 · A. 다른 장비 기준 완료


48시간 · B. 장비 자신의 과거 완료


48시간 · C. 장비 자신의 전체 기간 완료


### 3-1. AUC 비교

0.5는 무작위다. 묶음별 AUC와 평균, 그리고 장비별 AUC 중앙값을 함께 본다.

In [4]:
def tool_auc(W, name):
    y = (d.ttf_min <= W * HOUR).to_numpy().astype(int); sc = scores[(W, name)]
    out = {}
    for t, idx in d.groupby('Tool').indices.items():
        m = idx[ev_mask[idx]]
        if len(np.unique(y[m])) == 2:
            out[t] = roc_auc_score(y[m], sc[m])
    return pd.Series(out)

summ = []
for (W, name), _ in scores.items():
    r = res[(res.창 == f'{W}시간') & (res.기준 == name)]
    ta = tool_auc(W, name)
    summ.append({'창': f'{W}시간', '기준': name, **{f'묶음 {k}': round(v, 3) for k, v in zip(r.묶음, r.AUC)},
                 '평균 AUC': round(r.AUC.mean(), 3), '장비별 중앙값': round(ta.median(), 3),
                 'AUC>0.5 장비': f'{int((ta > .5).sum())}/{len(ta)}'})
summ = pd.DataFrame(summ)
display(summ)

fig = go.Figure()
for name, color in zip(['A. 다른 장비 기준', 'B. 장비 자신의 과거', 'C. 장비 자신의 전체 기간'], ['#9aa3ab', '#2a78d6', '#eb6834']):
    ta = tool_auc(24, name)
    fig.add_bar(x=ta.index, y=ta.values, name=name, marker_color=color)
fig.add_hline(y=0.5, line_dash='dash', annotation_text='무작위(0.5)')
fig.update_layout(title='그림 3-1. 장비별 AUC (24시간 창, 학습에 쓰지 않은 장비)', barmode='group', yaxis_title='AUC')
show(fig, 460)

,창,기준,묶음 1,묶음 2,묶음 3,묶음 4,평균 AUC,장비별 중앙값,AUC>0.5 장비
0,24시간,A. 다른 장비 기준,0.701,0.447,0.624,0.555,0.582,0.520,11/20
1,24시간,B. 장비 자신의 과거,0.567,0.402,0.672,0.492,0.533,0.525,12/20
2,24시간,C. 장비 자신의 전체 기간,0.620,0.445,0.527,0.500,0.523,0.525,12/20
3,48시간,A. 다른 장비 기준,0.657,0.401,0.553,0.514,0.531,0.494,9/20
4,48시간,B. 장비 자신의 과거,0.577,0.409,0.643,0.570,0.550,0.546,12/20
5,48시간,C. 장비 자신의 전체 기간,0.609,0.406,0.456,0.514,0.497,0.494,9/20


### 3-2. 중요한 피처 (B, 24시간 창)

LightGBM 정보 이득(gain)을 4묶음 평균하고 센서별로 합쳤다. 이득은 학습 데이터에서 잰 값이라, 처음 보는 장비에서도 쓸모 있다는 보장은 아니다.

In [5]:
imp = pd.concat(imps, axis=1).mean(axis=1)
by_s = imp.groupby(lambda c: next(s for s in SENSORS if c.startswith(s + '_'))).sum()
by_kind = imp.groupby(lambda c: '지난 24시간 평균' if c.endswith('_r24h') else ('지난 6시간 평균' if c.endswith('_r6h') else '현재 구간')).sum()
display((by_s / by_s.sum() * 100).round(1).sort_values(ascending=False).rename(index=NAMES).rename('비중(%)').to_frame())
display((by_kind / by_kind.sum() * 100).round(1).rename('비중(%)').to_frame())

,비중(%)
냉각 유량,16.1
냉각 압력,15.2
억제 전류,10.0
회전 각도,10.0
아르곤 유량(소스),7.9
챔버 압력,7.4
아르곤 유량(PBN),7.0
빔 전압,6.3
회전 속도,6.0
억제 전압,5.8


,비중(%)
지난 24시간 평균,76.4
지난 6시간 평균,15.7
현재 구간,7.8


## 4. 해석과 한계

- **장비 자신의 과거 기준(B)으로 바꿔도 거의 좋아지지 않았다.** 24시간 창 평균 AUC 0.533, 48시간 창 0.550이다. 다른 장비 기준(A)은 0.582·0.531이고, 미래 정보를 쓴 상한선(C)조차 0.523·0.497이다. 묶음별 편차가 크다(0.40~0.70).
- 중요 피처는 냉각 유량·냉각 압력이 위쪽이고, 지난 24시간 평균이 이득의 76%를 차지한다. 그래도 처음 보는 장비에서는 일관되게 통하지 않았다.

**진단: EDA에서 보인 신호가 모델에서 사라진 이유** (냉각 압력 z 하나만 쓴 별도 계산)

| 비교 | 장비별 AUC 중앙값 |
|---|---|
| 압력 저하 24시간 전 vs 평소 (EDA 방식, 전체 기간 기준) | 0.365 (0.5에서 0.135 떨어짐) |
| 3종 합친 라벨 24시간 전 vs 평소 | 0.437 |
| 압력 저하 24시간 전 vs 고장 직후 24시간 | 0.46 |
| **규칙(학습 없음)**: −냉각 압력 z, 장비 자신의 처음 60일 기준, 압력 저하 | 장비별 중앙값 **0.583** (19대 중 12대 0.5 초과), 장비를 합친 AUC 0.497 |

1. **고장 3종을 합치면 신호가 약해진다.** 냉각 압력은 압력 저하·누설 전에는 낮아지고 압력 과다 전에는 높아진다. 합친 라벨로는 방향이 섞인다.
2. **미래 정보를 빼면 약해진다.** 전체 기간 기준(0.365, 뒤집으면 0.635)이 처음 60일 기준에서는 0.583으로 줄어든다.
3. **장비마다 점수 수준이 달라서 합치면 사라진다.** 같은 장비 안에서는 순서가 맞아도(중앙값 0.583), 장비를 합쳐 한 기준으로 자르면 0.497이 된다. 장비마다 경보 기준을 따로 잡아야 한다는 뜻이다.
4. **장비의 3분의 1은 방향이 반대다.** 19대 중 7대는 0.5 아래라 하나의 규칙으로 모든 장비를 덮기 어렵다.
5. 고장 직후 구간도 고장 전과 비슷하게 낮다(0.46). 고장 기록이 몰려 있는 기간 전체에서 압력 수준이 낮은 것으로 보인다. 다만 음성 구간 중 고장 직후 비율은 4.9%라 큰 원인은 아니다.

**결론**: 냉각 압력에는 같은 장비 안에서 압력 저하 전에 낮아지는 **약한 신호(AUC 약 0.58)**가 있다. 하지만 장비마다 수준과 방향이 달라, 장비를 넘어 일반화하는 모델로 만들기는 어렵다.

**한계**: 기준 기간 60일은 임의로 정했다. 고장 종류별 모델과 장비별 경보 기준은 아직 시험하지 않았다. 사건 단위 평가(탐지율·헛경보)도 아직 하지 않았다.

**다음 후보**: (1) 압력 저하 하나만 장비 자기 기준으로 학습, (2) 장비마다 처음 60일 점수 분포로 경보 기준을 따로 정하고 사건 단위 탐지율·하루 헛경보로 평가.